# HW02 解答

## 2. 多层感知机 (MLP)

### 2.1 理论部分

#### 1. 线性模型级联证明
**问题**：证明两个线性模型的级联仍是一个线性模型。

**证明**：
给定：
$h = \mathbf{W}_1 \mathbf{x} + \mathbf{b}_1$
$o = \mathbf{W}_2 \mathbf{h} + \mathbf{b}_2$

将 $h$ 代入 $o$ 的表达式中：
$o = \mathbf{W}_2 (\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2$
$o = (\mathbf{W}_2 \mathbf{W}_1) \mathbf{x} + (\mathbf{W}_2 \mathbf{b}_1 + \mathbf{b}_2)$

令 $\mathbf{W}' = \mathbf{W}_2 \mathbf{W}_1$ 且 $\mathbf{b}' = \mathbf{W}_2 \mathbf{b}_1 + \mathbf{b}_2$。
则 $o = \mathbf{W}' \mathbf{x} + \mathbf{b}'$，这显然是一个线性模型的形式。

#### 2. 激活函数导数推导
- **Sigmoid 函数**: $\sigma(x) = \frac{1}{1 + e^{-x}}$
  导数：$\sigma'(x) = \frac{e^{-x}}{(1 + e^{-x})^2} = \frac{1}{1 + e^{-x}} \cdot \frac{e^{-x}}{1 + e^{-x}} = \sigma(x)(1 - \sigma(x))$

- **tanh 函数**: $\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$
  导数：$\tanh'(x) = 1 - \tanh^2(x)$

### 2.2 编程部分：从零实现 MLP
在此部分，我们将使用 PyTorch 实现一个简单的 MLP。

In [5]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

# 1. 加载 Fashion-MNIST
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_set = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
batch_size = 256
train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

# 2. 参数初始化
input_size = 28 * 28
hidden_size = 256
num_classes = 10

W1 = torch.randn(input_size, hidden_size) * 0.01
b1 = torch.zeros(hidden_size)
W2 = torch.randn(hidden_size, num_classes) * 0.01
b2 = torch.zeros(num_classes)

# 开启梯度
for p in [W1, b1, W2, b2]:
    p.requires_grad = True

# 3. ReLU 和 Softmax 交叉熵
def relu(x):
    return torch.maximum(x, torch.tensor(0.0))

def softmax_cross_entropy(logits, labels):
    # logits: (batch, num_classes), labels: (batch,)
    exp_logits = torch.exp(logits - logits.max(dim=1, keepdim=True)[0])
    probs = exp_logits / exp_logits.sum(dim=1, keepdim=True)
    loss = -torch.log(probs[range(len(labels)), labels] + 1e-8).mean()
    return loss

# 4. 训练循环（手动更新，使用 global 声明）
def train_epoch(lr=0.1):
    global W1, b1, W2, b2
    total_loss = 0
    for X, y in train_loader:
        X = X.view(X.size(0), -1)          # (batch, 784)
        h = relu(X @ W1 + b1)              # 隐藏层
        logits = h @ W2 + b2               # 输出层
        loss = softmax_cross_entropy(logits, y)
        total_loss += loss.item()
        
        loss.backward()                    # 计算梯度
        
        with torch.no_grad():
            # SGD 更新
            W1 -= lr * W1.grad
            b1 -= lr * b1.grad
            W2 -= lr * W2.grad
            b2 -= lr * b2.grad
            # 梯度清零
            W1.grad.zero_()
            b1.grad.zero_()
            W2.grad.zero_()
            b2.grad.zero_()
    return total_loss / len(train_loader)

def evaluate():
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in test_loader:
            X = X.view(X.size(0), -1)
            h = relu(X @ W1 + b1)
            logits = h @ W2 + b2
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total

# 5. 训练
epochs = 10
for epoch in range(epochs):
    loss = train_epoch(lr=0.1)
    acc = evaluate()
    print(f"Epoch {epoch+1:2d}, loss: {loss:.4f}, test acc: {acc:.4f}")

Epoch  1, loss: 0.7689, test acc: 0.7334
Epoch  2, loss: 0.4867, test acc: 0.8265
Epoch  3, loss: 0.4393, test acc: 0.8197
Epoch  4, loss: 0.4076, test acc: 0.8461
Epoch  5, loss: 0.3842, test acc: 0.8408
Epoch  6, loss: 0.3695, test acc: 0.8525
Epoch  7, loss: 0.3555, test acc: 0.8510
Epoch  8, loss: 0.3444, test acc: 0.8577
Epoch  9, loss: 0.3328, test acc: 0.8451
Epoch 10, loss: 0.3247, test acc: 0.8570


## 3. 模型选择与正则化

### 3.1 理论部分

#### 1. 训练误差与泛化误差
- **训练误差 (Training Error)**：模型在训练数据集上计算得到的误差。
- **泛化误差 (Generalization Error)**：模型应用在同样从原始样本分布中抽取的无限多新样本上时误差的期望。

#### 2. K 折交叉验证 (K-fold Cross-Validation)
将原始训练数据分成 K 个不重叠的子集。然后进行 K 次训练和验证，每次使用一个子集作为验证集，其余 K-1 个子集作为训练集。最后对这 K 次实验的误差取平均。

### 3.2 编程部分：Dropout 实现

In [2]:
# 在上一题代码基础上修改
def dropout_layer(X, dropout, is_training=True):
    if not is_training or dropout == 0:
        return X
    mask = (torch.rand(X.shape) > dropout).float()
    return X * mask / (1.0 - dropout)

# 修改参数更新，加入权重衰减 (L2 正则化)
def train_epoch_with_reg(lr=0.1, weight_decay=0.0, dropout_prob=0.0, is_training=True):
    total_loss = 0
    for X, y in train_loader:
        X = X.view(X.size(0), -1)
        # 隐藏层前向 + Dropout
        h = relu(X @ W1 + b1)
        h = dropout_layer(h, dropout_prob, is_training)
        logits = h @ W2 + b2
        loss = softmax_cross_entropy(logits, y)
        # 加入 L2 损失（手动加到 loss 上，但通常权重衰减在优化器中实现）
        if weight_decay > 0:
            l2_reg = weight_decay * (torch.sum(W1**2) + torch.sum(W2**2))
            loss = loss + l2_reg
        total_loss += loss.item()
        loss.backward()
        with torch.no_grad():
            # 权重衰减 = 旧权重乘以 (1 - lr * weight_decay) 然后再减梯度
            # 这里直接在更新中实现
            W1 -= lr * (W1.grad + weight_decay * W1)   # 等价于 (1 - lr*weight_decay)*W1 - lr*grad
            b1 -= lr * b1.grad
            W2 -= lr * (W2.grad + weight_decay * W2)
            b2 -= lr * b2.grad
            for p in [W1, b1, W2, b2]:
                if p.grad is not None:
                    p.grad.zero_()
    return total_loss / len(train_loader)

# 对比实验：设计高维多项式拟合或极少样本
# 采用极少样本（取前 100 个样本）训练一个复杂 MLP（隐藏层 512）
# 绘制三种情况的训练/验证误差曲线

## 4. 梯度消失与梯度爆炸

### 4.1 理论部分

#### 1. 原因分析
梯度消失和爆炸是由于深度网络中反向传播时的连乘效应导致的。如果每一层的梯度都小于 1，连乘后梯度会趋近于 0（消失）；如果大于 1，则会趋近于无穷大（爆炸）。

#### 2. ReLU 的作用
ReLU 在正区间的导数为 1，不会像 Sigmoid 在饱和区那样导数趋近于 0，因此能有效缓解梯度消失问题。

### 4.2 编程部分

In [3]:
import torch.nn as nn

# 1. 构建 20 层深层网络
class DeepNet(nn.Module):
    def __init__(self, activation='sigmoid', init_std=1.0):
        super().__init__()
        layers = []
        for i in range(20):
            linear = nn.Linear(256, 256)
            if init_std == 1.0:
                nn.init.normal_(linear.weight, mean=0, std=init_std)
            layers.append(linear)
            if activation == 'sigmoid':
                layers.append(nn.Sigmoid())
            else:
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

# 2. 模拟梯度消失（Sigmoid + std=1）
model = DeepNet(activation='sigmoid', init_std=1.0)
x = torch.randn(64, 256)
out = model(x)
loss = out.sum()
loss.backward()
# 打印各层梯度范数
for i, layer in enumerate(model.net):
    if isinstance(layer, nn.Linear):
        grad_norm = layer.weight.grad.norm().item()
        print(f"Layer {i} grad norm: {grad_norm}")   # 后几层会非常小

# 3. ReLU + std=10，可能 NaN
model2 = DeepNet(activation='relu', init_std=10.0)
x2 = torch.randn(64, 256)
out2 = model2(x2)
loss2 = out2.sum()
loss2.backward()  # 可能梯度爆炸，出现 inf 或 nan

# 4. Xavier 初始化 + ReLU
class DeepNetXavier(nn.Module):
    def __init__(self):
        super().__init__()
        layers = []
        for i in range(20):
            linear = nn.Linear(256, 256)
            nn.init.xavier_uniform_(linear.weight)
            layers.append(linear)
            layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

model3 = DeepNetXavier()
x3 = torch.randn(64, 256)
out3 = model3(x3)
loss3 = out3.sum()
loss3.backward()
# 梯度范数应稳定在合理范围如 [1e-6, 1e3]

Layer 0 grad norm: 10917.640625
Layer 2 grad norm: 7222.4013671875
Layer 4 grad norm: 6078.88134765625
Layer 6 grad norm: 4754.08056640625
Layer 8 grad norm: 4046.00439453125
Layer 10 grad norm: 2972.533203125
Layer 12 grad norm: 2321.451171875
Layer 14 grad norm: 2013.0301513671875
Layer 16 grad norm: 1424.0645751953125
Layer 18 grad norm: 1162.4619140625
Layer 20 grad norm: 853.009521484375
Layer 22 grad norm: 730.9367065429688
Layer 24 grad norm: 608.3202514648438
Layer 26 grad norm: 638.2435913085938
Layer 28 grad norm: 642.31982421875
Layer 30 grad norm: 564.4092407226562
Layer 32 grad norm: 533.2182006835938
Layer 34 grad norm: 629.3302612304688
Layer 36 grad norm: 575.5752563476562
Layer 38 grad norm: 580.1346435546875


## 5. 分布偏移 (Distribution Shift)

### 5.1 理论部分
- **协变量偏移 (Covariate Shift)**：输入 $P(x)$ 改变，但条件概率 $P(y|x)$ 不变。
- **标签偏移 (Label Shift)**：标签 $P(y)$ 改变，但 $P(x|y)$ 不变。

### 5.2 编程部分：重要性采样

In [ ]:
import sys
!{sys.executable} -m pip install scikit-learn scipy matplotlib -i https://pypi.tuna.tsinghua.edu.cn/simple --trusted-host pypi.tuna.tsinghua.edu.cn


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
     --------------------- ------------------ 4.7/8.9 MB 22.0 MB/s eta 0:00:01
     -------------------------------------- - 8.7/8.9 MB 21.5 MB/s eta 0:00:01
     ---------------------------------------- 8.9/8.9 MB 15.8 MB/s  0:00:00

   ------------- -------------------------- 1/3 [joblib]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   ------

In [4]:
import numpy as np
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_squared_error

# 1. 构造数据集
np.random.seed(42)
train_x = np.random.normal(-1, 1, (1000, 1))
train_y = 2 * train_x[:, 0] + np.random.normal(0, 0.1, 1000)

test_x = np.random.normal(2, 1, (500, 1))
test_y = 2 * test_x[:, 0] + np.random.normal(0, 0.1, 500)

# 2. 基线模型
base_model = LinearRegression()
base_model.fit(train_x, train_y)
pred_base = base_model.predict(test_x)
mse_base = mean_squared_error(test_y, pred_base)
print(f"Baseline MSE: {mse_base:.4f}")

# 3. 训练分类器区分训练集(0)和测试集(1)
X_combined = np.vstack([train_x, test_x])
y_combined = np.array([0]*1000 + [1]*500)
clf = LogisticRegression()
clf.fit(X_combined, y_combined)
prob_test = clf.predict_proba(train_x)[:, 1]  # P(test|x)

# 4. 计算权重 w_i ∝ P(test|x_i) / P(train|x_i) = prob_test / (1 - prob_test)
weights = prob_test / (1 - prob_test + 1e-8)
weights = weights / weights.sum() * len(weights)  # 归一化使得和等于样本数（可选）

# 5. 加权线性回归
def weighted_linear_regression(X, y, weights):
    # 加权最小二乘: W = (X^T diag(w) X)^{-1} X^T diag(w) y
    Xw = X * np.sqrt(weights).reshape(-1,1)
    yw = y * np.sqrt(weights)
    model = LinearRegression()
    model.fit(Xw, yw)
    return model

# 注意：X 需要加一列常数项，或者直接用 LinearRegression 拟合原始数据时传入 sample_weight
# 更简单：使用 sklearn 的 sample_weight
model_weighted = LinearRegression()
model_weighted.fit(train_x, train_y, sample_weight=weights)
pred_weighted = model_weighted.predict(test_x)
mse_weighted = mean_squared_error(test_y, pred_weighted)
print(f"Weighted MSE: {mse_weighted:.4f}")
print(f"Improvement: {mse_base - mse_weighted:.4f}")

Baseline MSE: 0.0102
Weighted MSE: 0.0240
Improvement: -0.0138
